# Repository Classifier — API Demo

This notebook demonstrates all public APIs and usage.

Pipeline: **Ground Truth → File Type Inference → Heuristic / LLM**.

## 1. Version & Imports

In [1]:
import repo_classifier

print(repo_classifier.__version__)

0.1.0


/Users/yichao/Projects/repo_classifier/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 2. Predefined: CLASSIFIERS, PHP / PYTHON / JAVASCRIPT

Built-in classifiers are `LanguageClassifier` subclasses (one instance per language). Use `CLASSIFIERS` for access by name or `PHP`, `PYTHON`, `JAVASCRIPT` directly. Legacy: `CLASSIFIER_NAMES`, `ALL_PROJECT_TYPES`, `DFT_PROJECT_TYPE_NAMES`.

In [2]:
from repo_classifier import CLASSIFIERS, PHP
from repo_classifier import CLASSIFIER_NAMES, ALL_PROJECT_TYPES, DFT_PROJECT_TYPE_NAMES

# Prefer: classifier instances (for classify_* and type hints)
print("CLASSIFIERS.names():", CLASSIFIERS.names())
print("CLASSIFIERS.php.name:", CLASSIFIERS.php.name)
print("PHP.project_type_names (sample):", PHP.project_type_names[:5])
print("CLASSIFIERS.php is PHP:", CLASSIFIERS.php is PHP)

# Legacy (derived from same built-ins)
print("CLASSIFIER_NAMES.all():", CLASSIFIER_NAMES.all())
print("ALL_PROJECT_TYPES keys:", list(ALL_PROJECT_TYPES.keys()))
print("DFT_PROJECT_TYPE_NAMES (php) sample:", list(DFT_PROJECT_TYPE_NAMES["php"])[:5])

CLASSIFIERS.names(): ['php', 'python', 'javascript']
CLASSIFIERS.php.name: php
PHP.project_type_names (sample): ['Web App', 'Framework', 'Framework Plugin', 'Framework Theme', 'Library']
CLASSIFIERS.php is PHP: True
CLASSIFIER_NAMES.all(): ['php', 'python', 'javascript']
ALL_PROJECT_TYPES keys: ['php', 'python', 'javascript']
DFT_PROJECT_TYPE_NAMES (php) sample: ['Web App', 'Framework', 'Framework Plugin', 'Framework Theme', 'Library']


## 3. Core: classify_repository_heuristic

Cascade: Ground Truth → File Type → Heuristic. No API key required.

In [3]:
from repo_classifier import classify_repository_heuristic, PHP

repo_url = "https://github.com/laravel/laravel"
# classifier: LanguageClassifier (e.g. PHP, CLASSIFIERS.php) or str ("php") or config dict
results = classify_repository_heuristic(repo_url, PHP, top_n=3)
print("Results:", results)

Results: {'Framework': 1.0, 'Library': 0.3466666666666667, 'Web App': 0.26666666666666666}


In [4]:
# Alternatively: by name (str) or inline config dict
results_by_name = classify_repository_heuristic(repo_url, "php", top_n=2)
custom_config = {
    "Web App": {"laravel": 10, "php": 5},
    "Framework": {"artisan": 10, "composer": 8},
}
results = classify_repository_heuristic(repo_url, custom_config, top_n=2)
print("Results (custom config):", results)

Results (custom config): {'Web App': 1.0, 'Framework': 0.03870967741935484}


## 4. Core: classify_repository_aimodel

Cascade: Ground Truth → File Type → LLM. Required: `model_name`, `api_key`. Use full model id: `provider/model` (e.g. `openai/gpt-4o`, `deepseek/deepseek-chat`). **Config from `.env`**: copy `docs/.env.example` to `docs/.env`, set `REPO_CLASSIFIER_API_KEY` and `REPO_CLASSIFIER_MODEL_NAME` (optional: `REPO_CLASSIFIER_TEMPERATURE`, `REPO_CLASSIFIER_TIMEOUT`).

In [5]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Load .env from docs/ (docs/.env when cwd is repo root, or .env when cwd is docs/)
if Path("docs/.env").exists():
    load_dotenv("docs/.env")
elif Path(".env").exists():
    load_dotenv(".env")

REPO_CLASSIFIER_API_KEY = os.getenv("REPO_CLASSIFIER_API_KEY")
REPO_CLASSIFIER_MODEL_NAME = os.getenv("REPO_CLASSIFIER_MODEL_NAME", "openai/gpt-4o")
REPO_CLASSIFIER_TEMPERATURE = float(os.getenv("REPO_CLASSIFIER_TEMPERATURE", "0.1"))
REPO_CLASSIFIER_TIMEOUT = int(os.getenv("REPO_CLASSIFIER_TIMEOUT", "60"))

In [6]:
from repo_classifier import classify_repository_aimodel, CLASSIFIERS

repo_url = "https://github.com/django/django"
# classifier: LanguageClassifier (e.g. CLASSIFIERS.python), or list of type names, or str ("python")
clf = CLASSIFIERS.python
if REPO_CLASSIFIER_API_KEY:
    results = classify_repository_aimodel(
        repo_url=repo_url,
        classifier=clf,
        model_name=REPO_CLASSIFIER_MODEL_NAME,
        api_key=REPO_CLASSIFIER_API_KEY,
        top_n=3,
        temperature=REPO_CLASSIFIER_TEMPERATURE,
        timeout=REPO_CLASSIFIER_TIMEOUT,
    )
    print("Results:", results)
else:
    print("Set REPO_CLASSIFIER_API_KEY in docs/.env (copy from docs/.env.example) to run.")

Results: {'Web Framework': 1.0, 'Library/Package': 0.0, 'Data Science': 0.0}


## 5. Registry: get_available_classifiers, get_classifier, register_classifier, unregister_classifier

Registry stores classifier **name → project_types dict**. Built-ins (php, python, javascript) are populated from `CLASSIFIERS`. `get_classifier("php")` returns the same dict as `CLASSIFIERS.php.project_types`.

In [7]:
from repo_classifier import (
    get_available_classifiers,
    get_classifier,
    register_classifier,
    unregister_classifier,
)

print("Available classifiers:", get_available_classifiers())
config = get_classifier("php")
print("get_classifier('php') project types (sample):", list(config.keys())[:5])

Available classifiers: ['php', 'python', 'javascript']
get_classifier('php') project types (sample): ['Web App', 'Framework', 'Framework Plugin', 'Framework Theme', 'Library']


In [8]:
# Register a custom classifier
custom = {
    "Game Engine": {"game": 10, "engine": 8},
    "Game Asset": {"sprite": 10, "texture": 8},
}
register_classifier("game_dev", custom)
print("After register_classifier('game_dev'):", get_available_classifiers())
print("get_classifier('game_dev'):", get_classifier("game_dev"))

unregister_classifier("game_dev")
print("After unregister_classifier('game_dev'):", get_available_classifiers())

After register_classifier('game_dev'): ['php', 'python', 'javascript', 'game_dev']
get_classifier('game_dev'): {'Game Engine': {'game': 10, 'engine': 8}, 'Game Asset': {'sprite': 10, 'texture': 8}}
After unregister_classifier('game_dev'): ['php', 'python', 'javascript']


## 6. Ground truth: add_ground_truth_entry, get_ground_truth_repos, load_ground_truth, save_ground_truth, evaluate_classifier

In [9]:
from repo_classifier import (
    add_ground_truth_entry,
    get_ground_truth_repos,
    load_ground_truth,
    save_ground_truth,
    evaluate_classifier,
)

add_ground_truth_entry("https://github.com/laravel/laravel", "Framework")
add_ground_truth_entry("https://github.com/django/django", "Web Framework")
print("get_ground_truth_repos():", get_ground_truth_repos())

get_ground_truth_repos(): {'https://github.com/laravel/laravel': 'Framework', 'https://github.com/django/django': 'Web Framework'}


In [10]:
# Save and load ground truth (JSON)
truth = get_ground_truth_repos()
save_ground_truth("/tmp/ground_truth_demo.json", truth)
loaded = load_ground_truth("/tmp/ground_truth_demo.json")
print("Loaded:", loaded)

Loaded: {'https://github.com/laravel/laravel': 'Framework', 'https://github.com/django/django': 'Web Framework'}


In [11]:
metrics = evaluate_classifier("php", get_ground_truth_repos())
print("evaluate_classifier('php', truth_dict):", metrics)

evaluate_classifier('php', truth_dict): {'accuracy': 0.0, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}


## 7. load_classifier_from_module, create_classifier_from_file

Load classifier from a Python module or from a config file.

In [12]:
from repo_classifier import load_classifier_from_module, create_classifier_from_file

# load_classifier_from_module(module_path, attribute_name=None) -> registered name
# create_classifier_from_file(file_path, encoding="utf-8") -> config dict
print("load_classifier_from_module(module_path, attribute_name=None) -> str")
print("create_classifier_from_file(file_path, encoding='utf-8') -> Dict")

load_classifier_from_module(module_path, attribute_name=None) -> str
create_classifier_from_file(file_path, encoding='utf-8') -> Dict
